# 2. Physics-Informed Structure Generation

Generate diverse atomic configurations for neural-network potential (NNP) training.

**Strategies covered:**
1. Einstein harmonic rattle (quantum + classical)
2. Normal-mode phonon sampling
3. Isotropic / deviatoric strain
4. Active learning via committee uncertainty

Each strategy is physically motivated — not random noise.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 2.1 Create a base structure

In [ ]:
from ase.build import fcc111
slab = fcc111('Cu', size=(2, 2, 3), vacuum=10.0, a=3.615)
print(f'Base structure: {len(slab)} atoms, Cu(111) 2×2×3')

## 2.2 Einstein Harmonic Rattle

Each atom is an independent quantum harmonic oscillator:

$$\sigma_i = \sqrt{\frac{\hbar}{2 m_i \omega} \coth\left(\frac{\hbar\omega}{2 k_B T}\right)}$$

- At high T: classical limit $\sigma \to \sqrt{k_B T / m\omega^2}$
- At low T: zero-point motion $\sigma \to \sqrt{\hbar / 2m\omega}$

In [ ]:
from science.generation.informed_sampler import EinsteinRattler

rattler = EinsteinRattler(omega_THz=5.0, quantum=True, rng_seed=42)

# Generate at different temperatures
temperatures = [100, 300, 600, 1000]
fig, axes = plt.subplots(1, len(temperatures), figsize=(14, 3))

for ax, T in zip(axes, temperatures):
    rattled = rattler.rattle(slab, T_K=T)
    disp = rattled.get_positions() - slab.get_positions()
    rmsd = np.sqrt(np.mean(disp**2))
    ax.hist(disp.flatten(), bins=30, alpha=0.7, color='steelblue')
    ax.set_title(f'T = {T} K\nRMSD = {rmsd:.3f} Å')
    ax.set_xlabel('Displacement (Å)')

plt.suptitle('Einstein Rattle: displacement distributions', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare quantum vs classical at low temperature
rattler_q = EinsteinRattler(omega_THz=5.0, quantum=True, rng_seed=1)
rattler_c = EinsteinRattler(omega_THz=5.0, quantum=False, rng_seed=1)

T_range = np.linspace(1, 1000, 50)
sigma_q = [rattler_q._sigma(63.546, T) for T in T_range]
sigma_c = [rattler_c._sigma(63.546, T) for T in T_range]

plt.figure(figsize=(7, 4))
plt.plot(T_range, sigma_q, 'b-', lw=2, label='Quantum (full coth)')
plt.plot(T_range, sigma_c, 'r--', lw=2, label='Classical (√kT/mω²)')
plt.axhline(sigma_q[0], color='gray', ls=':', alpha=0.5, label=f'ZPE floor = {sigma_q[0]:.4f} Å')
plt.xlabel('Temperature (K)'); plt.ylabel('σ (Å)')
plt.title('Displacement σ for Cu (m=63.5 amu, ω=5 THz)')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 2.3 Strain Sampling
Deform the cell to sample the equation of state (EOS) and elastic response.

In [ ]:
from science.generation.informed_sampler import strain_sample

strained = strain_sample(slab, strain_max=0.06, n=8, mode='isotropic', rng_seed=42)
print(f'Generated {len(strained)} strained structures')

# Show cell volume variation
vols = [s.get_volume() for s in strained]
v0 = slab.get_volume()
plt.figure(figsize=(6, 3))
plt.bar(range(len(vols)), [(v/v0 - 1)*100 for v in vols], color='coral')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('Structure index'); plt.ylabel('Volume change (%)')
plt.title('Isotropic strain sampling (±6%)')
plt.show()

## 2.4 Normal-Mode (Phonon) Sampling

If a Hessian is available, we can sample along phonon eigenmodes with equipartition amplitudes:

$$A_\nu = \sqrt{k_B T / \lambda_\nu}$$

In [ ]:
from science.generation.informed_sampler import NormalModeSampler

# Create a synthetic Hessian (diagonal, typical spring constants)
N = len(slab)
rng = np.random.default_rng(42)
# Eigenvalues in eV/Å² (typical: 1-20 for metals)
eigenvalues = rng.uniform(0.5, 15.0, 3 * N)
eigenvalues[:3] = 0.0  # acoustic modes
hessian = np.diag(eigenvalues)  # simplified diagonal Hessian

sampler = NormalModeSampler(rng_seed=42)
sampler.fit(hessian, slab.get_masses())

freqs = sampler.mode_frequencies_THz()
print(f'Optical mode frequencies: {freqs[:5].round(1)} ... {freqs[-3:].round(1)} THz')
print(f'Total modes: {len(freqs)} (of {3*N} total, {3*N - len(freqs)} acoustic removed)')

In [ ]:
# Generate batch at T=500K
configs = sampler.generate_batch(slab, T_K=500, n=20)

# Measure displacement distribution
all_disp = []
for c in configs:
    d = c.get_positions() - slab.get_positions()
    all_disp.append(np.linalg.norm(d, axis=1).mean())

plt.figure(figsize=(6, 3))
plt.hist(all_disp, bins=15, color='mediumpurple', alpha=0.8)
plt.xlabel('Mean atomic displacement (Å)')
plt.ylabel('Count')
plt.title(f'Normal-mode sampling at T=500K ({len(configs)} structures)')
plt.show()

## 2.5 Active Learning: Committee Uncertainty

Select the most informative structures for DFT by measuring model disagreement.

In [ ]:
from science.generation.informed_sampler import CommitteeUncertaintySampler

# Simulate 3 committee models with different noise
rng = np.random.default_rng(42)
def make_model(bias, noise):
    def predict(atoms):
        pos = atoms.get_positions()
        return -100.0 + bias + rng.normal(0, noise) * len(pos)
    return predict

committee = [make_model(0.0, 0.01), make_model(0.1, 0.02), make_model(-0.05, 0.015)]
al_sampler = CommitteeUncertaintySampler(committee, threshold=0.005)

# Generate candidates
candidates = rattler.generate_batch(slab, T_K=600, n=50)

# Split into DFT-needed and well-covered
needs_dft, covered = al_sampler.filter_above_threshold(candidates)
print(f'Candidates: {len(candidates)}')
print(f'Needs DFT (σ > 0.005 eV/atom): {len(needs_dft)}')
print(f'Well-covered (skip DFT): {len(covered)}')
print(f'DFT savings: {len(covered)/len(candidates)*100:.0f}%')

In [ ]:
# Visualise uncertainty distribution
uncertainties = [al_sampler.uncertainty(c) for c in candidates]
plt.figure(figsize=(6, 3))
plt.hist(uncertainties, bins=20, color='tomato', alpha=0.7)
plt.axvline(0.005, color='k', ls='--', label=f'threshold = 0.005 eV/atom')
plt.xlabel('Per-atom uncertainty (eV/atom)')
plt.ylabel('Count')
plt.title('Committee disagreement across candidates')
plt.legend()
plt.show()